In [1]:
import pickle
import pandas as pd

from sklearn.model_selection import RandomizedSearchCV, GridSearchCV

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

import warnings
warnings.filterwarnings("ignore")

In [6]:
x_train = pickle.load(open("../models/x_train_cls.pkl","rb"))
x_test = pickle.load(open("../models/x_test_cls.pkl","rb"))

y_train = pickle.load(open("../models/y_train_cls.pkl","rb"))
y_test = pickle.load(open("../models/y_test_cls.pkl","rb"))

In [7]:
# Parameter Grid

log_params = {
    "C":[0.001, 0.01, 0.1, 1, 10, 100],
    "solver":["lbfgs", "liblinear"],
    "penalty":["l2"],
    "max_iter":[100, 300, 500]
}

gb_params = {
    "n_estimators":[100, 200, 300],
    "learning_rate":[0.01, 0.05, 0.1],
    "max_depth":[3, 5, 7],
    "subsample":[0.8, 0.9, 1.0],
    "min_samples_split":[2, 5, 10]
}

In [9]:
# Logistic Regression (GridSearchCV)

log_search = GridSearchCV(
    estimator=LogisticRegression(),
    param_grid=log_params,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=2
)

log_search.fit(x_train, y_train)
best_log = log_search.best_estimator_

print("Best Logistic Parameters:")
print(log_search.best_params_)

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best Logistic Parameters:
{'C': 0.001, 'max_iter': 100, 'penalty': 'l2', 'solver': 'lbfgs'}


In [10]:
# Gradient Boosting (RandomizedSearchCV)

gb_search = RandomizedSearchCV(
    estimator=GradientBoostingClassifier(random_state=42),
    param_distributions=gb_params,
    n_iter=20,
    cv=3,
    scoring="f1",
    random_state=42,
    n_jobs=-1,
    verbose=2
)

gb_search.fit(x_train, y_train)
best_gb = gb_search.best_estimator_

print("Best Gradient Boosting Parameters:")
print(gb_search.best_params_)

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best Gradient Boosting Parameters:
{'subsample': 0.8, 'n_estimators': 100, 'min_samples_split': 5, 'max_depth': 3, 'learning_rate': 0.01}


In [11]:
# Evaluation Function

def evaluate_model(model, x_test, y_test):
    prediction = model.predict(x_test)
    return {
        "Accuracy": accuracy_score(y_test, prediction),
        "Precision": precision_score(y_test, prediction),
        "Recall": recall_score(y_test, prediction),
        "F1 Score": f1_score(y_test, prediction),
        "ROC AUC": roc_auc_score(y_test, prediction)
    }

In [12]:
# Evaluate Both Models

log_result = evaluate_model(best_log, x_test, y_test)
gb_result = evaluate_model(best_gb, x_test, y_test)

In [21]:
# Compare Results

comparison = pd.DataFrame({
    "Model":["Logistic Regression", "Gradient Boosting"],
    "Accuracy":[log_result["Accuracy"], gb_result["Accuracy"]],
    "Precision":[log_result["Precision"], gb_result["Precision"]],
    "Recall":[log_result["Recall"], gb_result["Recall"]],
    "F1 Score":[log_result["F1 Score"], gb_result["F1 Score"]],
    "ROC AUC":[log_result["ROC AUC"], gb_result["ROC AUC"]],
})

comparison.sort_values(
    by="Accuracy",
    ascending=False,
    inplace=True
)

comparison

,Model,Accuracy,Precision,Recall,F1 Score,ROC AUC
0,Logistic Regression,0.56965,0.577547,0.781216,0.664117,0.548930
1,Gradient Boosting,0.55660,0.554579,0.944087,0.698716,0.518651


In [ ]:
# Save Comparison
comparison.to_csv("../models/hyperparameter_results.csv", index=False)

In [23]:
# Select Final Model

if log_result["Accuracy"] >= gb_result["Accuracy"]:
    final_model = best_log
    selected_model = "Logistic Regression"
else:
    final_model = best_gb
    selected_model = "Gradient Boosting"

print(f"Final Selected Model : {selected_model}")

Final Selected Model : Logistic Regression


In [24]:
# Save Tuned Models

pickle.dump(
    best_log,
    open("../models/logistic_tuned.pkl","wb")
)

pickle.dump(
    best_gb,
    open("../models/gradient_boosting_tuned.pkl","wb")
)

print("Both tuned models saved successfully.")

Both tuned models saved successfully.
